In [ ]:
# =========================
# CELL 1: setup + install
# =========================

from google.colab import drive

drive.mount("/content/drive")

import os
import sys
import json
import time
import glob
import shutil
import random
import shlex
import subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/moses_full_reproduction")
REPO_DIR = Path("/content/MoSEs")
REPO_CACHE = DRIVE_ROOT / "repo_cache"
HF_CACHE = DRIVE_ROOT / "hf_cache"
RESULTS_ROOT = DRIVE_ROOT / "final_outputs"

for p in [DRIVE_ROOT, REPO_CACHE, HF_CACHE, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def run_cmd(cmd, cwd=None, env=None, shell=False, check=True):
    printable = (
        cmd if isinstance(cmd, str) else " ".join(shlex.quote(str(x)) for x in cmd)
    )
    print("\n" + "=" * 100)
    print(printable)
    print("=" * 100)
    return subprocess.run(cmd, cwd=cwd, env=env, shell=shell, check=check)


def copytree_merge(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        s = item
        d = dst / item.name
        if s.is_dir():
            copytree_merge(s, d)
        else:
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(s, d)


# fresh clone
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd(["git", "clone", "https://github.com/creator-xi/MoSEs.git", str(REPO_DIR)])

# restore cached outputs if they exist
for rel in ["weights", "logs"]:
    copytree_merge(REPO_CACHE / rel, REPO_DIR / rel)

# restore previously generated split_datasets* folders
cached_data_dir = REPO_CACHE / "data"
if cached_data_dir.exists():
    for p in cached_data_dir.glob("split_datasets*"):
        copytree_merge(p, REPO_DIR / "data" / p.name)

# install dependencies
run_cmd(
    [sys.executable, "-m", "pip", "uninstall", "-y", "transformers", "FlagEmbedding"],
    check=False,
)
run_cmd(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==4.44.2",
        "FlagEmbedding==1.2.10",
        "accelerate<1.0",
        "sentencepiece",
    ]
)
run_cmd(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas",
        "numpy",
        "scikit-learn",
        "xgboost",
        "tqdm",
        "openai",
        "datasets",
        "matplotlib",
        "joblib",
        "scipy",
    ]
)
run_cmd(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
)

run_cmd(["nvidia-smi"], check=False)
print("Setup complete.")

In [ ]:
# =========================
# CELL 2: configuration + helpers
# =========================

import json
import pandas as pd
from pathlib import Path

DATA_INPUT_DIR = REPO_DIR / "data" / "doc4split"

MAIN_KEYS = ["cmv", "sci", "wp", "xsum"]
LOW_KEYS = ["cnn", "dialogsum", "imdb", "pubmedqa"]

# You can flip any of these to False if you need to rerun only part of the pipeline.
RUN_FAST = True
RUN_LASTDE = True
RUN_ROBERTA = True

# Official example scripts clearly show main fast/lastde use *_train as reference set.
# The released roberta example points to the unsplit folder; keep this toggle if you want to match that behavior.
ROBERTA_MAIN_USE_FULL_FOLDER = True

DETECTORS = {
    "fast": {
        "enabled": RUN_FAST,
        "crit_type": "fast",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_fast",
        "extra_args": [],
    },
    "lastde": {
        "enabled": RUN_LASTDE,
        "crit_type": "lastde",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_lastde",
        "extra_args": ["--embed_size", "4", "--epsilon", "8", "--tau_prime", "15"],
    },
    "roberta_base": {
        "enabled": RUN_ROBERTA,
        "crit_type": "roberta",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_roberta_base",
        "extra_args": ["--roberta_model_name", "roberta-base-openai-detector"],
    },
}


def sync_outputs_to_drive():
    (REPO_CACHE / "data").mkdir(parents=True, exist_ok=True)
    for p in (REPO_DIR / "data").glob("split_datasets*"):
        copytree_merge(p, REPO_CACHE / "data" / p.name)
    copytree_merge(REPO_DIR / "weights", REPO_CACHE / "weights")
    copytree_merge(REPO_DIR / "logs", REPO_CACHE / "logs")


def dataset_key_from_json_name(name):
    key = Path(name).stem.lower()
    if key.endswith("_dataset"):
        key = key[:-8]
    return key


def subset_json_folder(src_dir, dst_dir, allowed_keys):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in src_dir.glob("*.json"):
        if dataset_key_from_json_name(p.name) in allowed_keys:
            shutil.copy2(p, dst_dir / p.name)


def split_json_folder_dataset_aware(
    src_dir, train_dir, test_dir, main_test_per_label=50, seed=42
):
    src_dir = Path(src_dir)
    train_dir = Path(train_dir)
    test_dir = Path(test_dir)
    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)

    for p in sorted(src_dir.glob("*.json")):
        key = dataset_key_from_json_name(p.name)
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)

        label0 = [x for x in data if int(x["label"]) == 0]
        label1 = [x for x in data if int(x["label"]) == 1]
        rng.shuffle(label0)
        rng.shuffle(label1)

        if key in LOW_KEYS:
            # robust low-resource split: half per label to test, half to reference
            test_n = min(len(label0), len(label1)) // 2
        else:
            # main setting: follow released example style with 50 per label when possible
            test_n = min(main_test_per_label, len(label0) // 2, len(label1) // 2)

        test_data = label0[:test_n] + label1[:test_n]
        train_data = label0[test_n:] + label1[test_n:]

        with open(test_dir / p.name, "w", encoding="utf-8") as f:
            json.dump(test_data, f, ensure_ascii=False, indent=2)
        with open(train_dir / p.name, "w", encoding="utf-8") as f:
            json.dump(train_data, f, ensure_ascii=False, indent=2)

        print(
            f"{p.name}: train={len(train_data)}, test={len(test_data)}, test_per_label={test_n}"
        )


def prepare_detector_views(processed_dir):
    processed_dir = Path(processed_dir)

    main_dir = Path(str(processed_dir) + "_main")
    low_dir = Path(str(processed_dir) + "_low")
    main_train = Path(str(processed_dir) + "_main_train")
    main_test = Path(str(processed_dir) + "_main_test")
    low_train = Path(str(processed_dir) + "_low_train")
    low_test = Path(str(processed_dir) + "_low_test")

    # subset
    if not main_dir.exists() or not list(main_dir.glob("*.json")):
        subset_json_folder(processed_dir, main_dir, MAIN_KEYS)
    if not low_dir.exists() or not list(low_dir.glob("*.json")):
        subset_json_folder(processed_dir, low_dir, LOW_KEYS)

    # split
    if (
        not main_train.exists()
        or not list(main_train.glob("*.json"))
        or not main_test.exists()
        or not list(main_test.glob("*.json"))
    ):
        split_json_folder_dataset_aware(
            main_dir, main_train, main_test, main_test_per_label=50, seed=42
        )
    if (
        not low_train.exists()
        or not list(low_train.glob("*.json"))
        or not low_test.exists()
        or not list(low_test.glob("*.json"))
    ):
        split_json_folder_dataset_aware(
            low_dir, low_train, low_test, main_test_per_label=50, seed=42
        )

    return {
        "full": processed_dir,
        "main": main_dir,
        "low": low_dir,
        "main_train": main_train,
        "main_test": main_test,
        "low_train": low_train,
        "low_test": low_test,
    }


def flatten_json(x, prefix=""):
    out = {}
    if isinstance(x, dict):
        for k, v in x.items():
            out.update(flatten_json(v, f"{prefix}{k}." if prefix else f"{k}."))
    elif isinstance(x, list):
        if len(x) <= 20:
            out[prefix[:-1]] = str(x)
    else:
        out[prefix[:-1]] = x
    return out


print("Helpers ready.")

In [ ]:
# =========================
# CELL 3: verify input data + build processed folders for all detectors
# =========================

csvs = sorted(DATA_INPUT_DIR.glob("*.csv"))
print("Found CSV files:")
for p in csvs:
    print(" -", p.name)

assert len(csvs) > 0, f"No CSV files found in {DATA_INPUT_DIR}"


def build_processed_folder(det_name, cfg):
    out_dir = REPO_DIR / "data" / cfg["out_name"]
    jsons = list(out_dir.glob("*.json")) if out_dir.exists() else []
    if len(jsons) >= 8:
        print(f"[SKIP] {det_name}: found {len(jsons)} processed JSONs in {out_dir}")
        return out_dir

    cmd = [
        sys.executable,
        "split_datasets.py",
        "--input_directory",
        str(DATA_INPUT_DIR),
        "--output_directory",
        str(out_dir),
        "--embedding_type",
        "encode",
        "--embedding_model_name",
        "BAAI/bge-m3",
        "--scoring_model_name",
        "EleutherAI/gpt-neo-2.7B",
        "--batch_size",
        "500",
        "--crit_type",
        cfg["crit_type"],
        "--cache_dir",
        str(HF_CACHE),
    ] + cfg["extra_args"]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()
    return out_dir


PROCESSED_DIRS = {}

for det_name, cfg in DETECTORS.items():
    if not cfg["enabled"]:
        continue
    PROCESSED_DIRS[det_name] = build_processed_folder(det_name, cfg)

print("\nProcessed dirs:")
for k, v in PROCESSED_DIRS.items():
    print(k, "->", v)

In [ ]:
# =========================
# CELL 4: create main/low-resource subsets and train/test views
# =========================

VIEW_PATHS = {}

for det_name, processed_dir in PROCESSED_DIRS.items():
    print(f"\nPreparing views for {det_name}")
    VIEW_PATHS[det_name] = prepare_detector_views(processed_dir)

sync_outputs_to_drive()

for det_name, views in VIEW_PATHS.items():
    print(f"\n=== {det_name} ===")
    for k, v in views.items():
        count = len(list(Path(v).glob("*.json"))) if Path(v).exists() else 0
        print(f"{k:>10}: {v} ({count} json files)")

In [ ]:
# =========================
# CELL 5: train SAR for main and low-resource
# =========================

SAR_MAIN_OUT = REPO_DIR / "weights" / "model_output_main_c16"
SAR_LOW_OUT = REPO_DIR / "weights" / "model_output_low_c16"


def train_sar_if_needed(dataset_folder, output_dir):
    weight_path = output_dir / "subcentroids_head_epoch100.pt"
    class_path = output_dir / "class_names.json"

    if weight_path.exists() and class_path.exists():
        print(f"[SKIP] SAR already exists at {output_dir}")
        return weight_path, class_path

    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "SAR.py",
        "train",
        "--model_name",
        "BAAI/bge-m3",
        "--datasets_folder",
        str(dataset_folder),
        "--embedding_type",
        "encode",
        "--num_epochs",
        "100",
        "--batch_size",
        "32",
        "--num_subcentroids",
        "4",
        "--output_dir",
        str(output_dir),
    ]
    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()
    return weight_path, class_path


# The released repo examples train SAR on the fast processed data for the main setting.
# Reuse that idea here for main and low-resource.
assert (
    "fast" in VIEW_PATHS
), "fast detector processed folder is needed to train SAR as in the released examples."

SAR_MAIN_PATH, SAR_MAIN_CLASS = train_sar_if_needed(
    VIEW_PATHS["fast"]["main"], SAR_MAIN_OUT
)
SAR_LOW_PATH, SAR_LOW_CLASS = train_sar_if_needed(
    VIEW_PATHS["fast"]["low"], SAR_LOW_OUT
)

print("Main SAR:", SAR_MAIN_PATH, SAR_MAIN_CLASS)
print("Low  SAR:", SAR_LOW_PATH, SAR_LOW_CLASS)

In [ ]:
# =========================
# CELL 6: run CTE evaluation for all detectors on main + low-resource
# =========================

DONE_DIR = REPO_DIR / "logs" / "done_markers"
DONE_DIR.mkdir(parents=True, exist_ok=True)


def run_cte_folder(
    test_folder, datasets_folder, result_prefix, sar_path, class_path, marker_prefix
):
    test_folder = Path(test_folder)
    result_prefix = Path(result_prefix)
    result_prefix.parent.mkdir(parents=True, exist_ok=True)

    for test_file in sorted(test_folder.glob("*.json")):
        marker = DONE_DIR / f"{marker_prefix}__{test_file.stem}.done"
        if marker.exists():
            print(f"[SKIP] {marker.name}")
            continue

        cmd = [
            sys.executable,
            "CTE.py",
            "--file_path",
            str(test_file),
            "--result_file_path",
            str(result_prefix),
            "--datasets_folder",
            str(datasets_folder),
            "--sar_path",
            str(sar_path),
            "--class_path",
            str(class_path),
        ]
        run_cmd(cmd, cwd=REPO_DIR)
        marker.touch()
        sync_outputs_to_drive()


for det_name, views in VIEW_PATHS.items():
    print(f"\n==================== {det_name}: MAIN ====================")

    if det_name == "roberta_base" and ROBERTA_MAIN_USE_FULL_FOLDER:
        main_ref_folder = views["main"]
    else:
        main_ref_folder = views["main_train"]

    run_cte_folder(
        test_folder=views["main_test"],
        datasets_folder=main_ref_folder,
        result_prefix=REPO_DIR / "logs" / "reproduction" / "main" / det_name,
        sar_path=SAR_MAIN_PATH,
        class_path=SAR_MAIN_CLASS,
        marker_prefix=f"main__{det_name}",
    )

    print(f"\n==================== {det_name}: LOW ====================")
    run_cte_folder(
        test_folder=views["low_test"],
        datasets_folder=views["low_train"],
        result_prefix=REPO_DIR / "logs" / "reproduction" / "low" / det_name,
        sar_path=SAR_LOW_PATH,
        class_path=SAR_LOW_CLASS,
        marker_prefix=f"low__{det_name}",
    )

print("All requested CTE runs finished.")
sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 7: collect outputs into a summary table + save to Drive
# =========================

records = []

for p in sorted((REPO_DIR / "logs").rglob("*.json")):
    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
        flat = flatten_json(obj)
        rec = {"file": str(p.relative_to(REPO_DIR))}
        for k, v in flat.items():
            if isinstance(v, (int, float, str)):
                rec[k] = v
        records.append(rec)
    except Exception as e:
        records.append({"file": str(p.relative_to(REPO_DIR)), "parse_error": str(e)})

df = pd.DataFrame(records).fillna("")
summary_csv = RESULTS_ROOT / "reproduction_log_summary.csv"
df.to_csv(summary_csv, index=False)

print(f"Saved summary to: {summary_csv}")
display(df.head(50))

# also make a zip of logs and weights
logs_zip = shutil.make_archive(
    str(RESULTS_ROOT / "moses_logs"), "zip", REPO_DIR / "logs"
)
weights_zip = shutil.make_archive(
    str(RESULTS_ROOT / "moses_weights"), "zip", REPO_DIR / "weights"
)

print("Logs zip   :", logs_zip)
print("Weights zip:", weights_zip)

sync_outputs_to_drive()
print("Everything synced back to Drive.")